### Import libraries

All imports for the whole notebook live in this single cell. If you run a later cell and get a `NameError`, come back and re-run this one.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Machine learning
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

from scipy.linalg import inv, sqrtm

# Qiskit (only needed for Track 2D real-hardware comparison)
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import ParameterVector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import (
    Batch,
    EstimatorOptions,
    EstimatorV2 as Estimator,
    QiskitRuntimeService,
)

import tqdm

# Global random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
def download_pqk_dataset(data_dir="data_tutorial/pqk"):
    """Download the four CSV files from the Qiskit documentation repo."""
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    base_url = (
        "https://raw.githubusercontent.com/Qiskit/documentation/main/"
        "datasets/tutorials/pqk"
    )
    files = [
        "noisy_data.csv",
        "noisy_quantum_data.csv",
        "projections_test.csv",
        "projections_train.csv",
        "test_data.csv",
        "noisy_data.csv"
    ]

    for filename in files:
        url = f"{base_url}/{filename}"
        dest = data_dir / filename
        print(f"  {filename} ...", end=" ", flush=True)
        urllib.request.urlretrieve(url, dest)
        print(f"✓ ({dest.stat().st_size:,} bytes)")

    print(f"\nAll files saved to {data_dir}/")
    return data_dir


DATA_DIR = download_pqk_dataset()

## 2. IBM Quantum Account Setup

> **Team account**
>
> For this hackathon you'll use a shared **team account** with 25 minutes of QPU time. To access it you need an API key which you received today.
>
> If your team runs out of QPU time, contact your IBM mentor (Sophy) or a UW admin (Kristen)

> **⚠️ Back up your results before May 16, 2026 ⚠️**
>
> The hackathon team accounts will be deleted on May 16th, 2026. Save your job results to disk before then — see [Retrieve job results](https://quantum.cloud.ibm.com/docs/en/guides/save-jobs#retrieve-job-results-from-ibm-quantum) and [Save results to disk](https://quantum.cloud.ibm.com/docs/en/guides/save-jobs#save-results-to-disk).

In [ ]:
API_KEY = "YOUR_API_KEY_HERE"

# Uncomment to save credentials once (then comment back out):
# QiskitRuntimeService.save_account(
#     token=API_KEY,
#     overwrite=True,
# )

# Uncomment to load and verify your service connection:
# service = QiskitRuntimeService(token=API_KEY)
# print("Available backends:", [b.name for b in service.backends()])

In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('data_tutorial/pqk/noisy_data.csv')

# Isolate only the integer columns to avoid adding noise to text/floats
int_cols = df.select_dtypes(include=['int', 'int64']).columns

# Define noise parameters
mu, sigma = 0, 2.5 

# Generate noise matching the shape of integer columns
noise = np.random.normal(mu, sigma, df[int_cols].shape)

# Add noise, round to nearest whole number, and cast back to integers
df[int_cols] = np.rint(df[int_cols] + noise).astype(int)

# Save result
df.to_csv('data_tutorial/pqk/noisy_data.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np

# 1. Load your CSV file
df = pd.read_csv('data_tutorial/pqk/noisy_quantum_data.csv')

# 2. Define noise parameters (Mean and Standard Deviation)
mu, sigma = 0, 2.5

# 3. Generate noise with the same shape as your dataframe
# Only apply to numerical columns to avoid errors with text
noise = np.random.normal(mu, sigma, df.shape)

# 4. Add the noise to the original data
df_noisy = df + noise

# 5. Save the result back to a CSV
df_noisy.to_csv('data_tutorial/pqk/noisy_quantum_data.csv', index=False)

In [ ]:
# ---------------------------------------------------------------------------
# Helper: sanitize CSV files into well-formed ones.
#
# The data files in this dataset sometimes ship with two quirks that confuse pd.read_csv:
#   - a UTF-8 BOM at the start
#   - every line wrapped in an extra pair of double quotes, so the whole row is parsed as one column of 172/74 strings (motif CSVs only)
#   - whitespace-separated floats instead of commas (projection CSVs only)
#
# We detect and repair these in place so the IBM tutorial code below works unmodified. Already well-formed files are left alone.
# ---------------------------------------------------------------------------
def sanitize_csv_files(data_dir):
    """Normalize quirky CSV files to well-formed UTF-8 comma-separated CSVs."""
    data_dir = Path(data_dir)

    # Motif files: strip BOM and outer line-wrap quotes if present.
    for name in ["noisy_data.csv", "test_data.csv"]:
        path = data_dir / name
        if not path.exists():
            continue
        with open(path, "rb") as f:
            raw = f.read()
        text = raw.decode("utf-8-sig")
        lines = text.replace("\r\n", "\n").strip().split("\n")
        wrapped = all(
            len(L) >= 2 and L.startswith('"') and L.endswith('"') for L in lines
        )
        if wrapped:
            lines = [L[1:-1] for L in lines]
        cleaned = "\n".join(lines) + "\n"
        if cleaned.encode("utf-8") != raw:
            path.write_text(cleaned, encoding="utf-8")
            print(f"  Sanitized {name}")

    # Projection files: convert whitespace-separated to comma-separated if needed.
    for name in ["projections_train.csv", "projections_test.csv"]:
        path = data_dir / name
        if not path.exists():
            continue
        with open(path, "rb") as f:
            head = f.read(2048)
        text_head = head.decode("utf-8-sig", errors="replace")
        first_line = text_head.split("\n")[0]
        # If the first line has many whitespace-separated floats and no commas,
        # it\'s the whitespace-separated format.
        if "," not in first_line and len(first_line.split()) > 10:
            # Whitespace-separated → reload with pandas sep=r"\s+" and rewrite as CSV.
            from io import StringIO
            with open(path, "rb") as f:
                text = f.read().decode("utf-8-sig")
            df = pd.read_csv(StringIO(text), sep=r"\s+", header=None, engine="python")
            df.to_csv(path, index=False, header=False)
            print(f"  Sanitized {name}")


sanitize_csv_files(DATA_DIR)


# ---------------------------------------------------------------------------
# The two preprocessing functions below are the IBM PQK tutorial code, 
# kept intentionally close to the original so your results align with the paper (Utro et al. 2025) and the tutorial. See:
#   https://quantum.cloud.ibm.com/docs/en/tutorials/projected-quantum-kernels
# ---------------------------------------------------------------------------
def preprocess_data(dir_root, args):
    """
    Preprocess the training and test data.
    Tutorial-exact: load CSVs, normalize motif ID 17 → 14, force canonical
    column names, optionally filter by spacer-motif rules, binarize labels
    at the cytotoxicity threshold, shift motif IDs to start at 0.
    """
    # Read from the csv files
    noisy_data = pd.read_csv(
        os.path.join(dir_root, args["file_noisy_data"]),
        encoding="unicode_escape",
        sep=",",
    )
    test_data = pd.read_csv(
        os.path.join(dir_root, args["file_test_data"]),
        encoding="unicode_escape",
        sep=",",
    )

    # Fix the last motif ID
    noisy_data[noisy_data == 17] = 14
    noisy_data.columns = [
        "Cell Number",
        "motif",
        "motif.1",
        "motif.2",
        "motif.3",
        "motif.4",
        "Nalm 6 Cytotoxicity",
    ]
    test_data[test_data == 17] = 14
    test_data.columns = [
        "Cell Number",
        "motif",
        "motif.1",
        "motif.2",
        "motif.3",
        "motif.4",
        "Nalm 6 Cytotoxicity",
    ]

    # Adjust motif at the third position
    if args["filter_for_spacer_motif_third_position"]:
        noisy_data = noisy_data[
            (noisy_data["motif.2"] == 14) | (noisy_data["motif.2"] == 0)
        ]
        test_data = test_data[
            (test_data["motif.2"] == 14) | (test_data["motif.2"] == 0)
        ]

    noisy_data = noisy_data[
        args["motifs_to_use"] + [args["label_name"], "Cell Number"]
    ]
    test_data = test_data[
        args["motifs_to_use"] + [args["label_name"], "Cell Number"]
    ]

    # Adjust motif at the last position
    if not args["allow_spacer_motif_last_position"]:
        last_motif = args["motifs_to_use"][len(args["motifs_to_use"]) - 1]
        noisy_data = noisy_data[
            (noisy_data[last_motif] != 14) & (noisy_data[last_motif] != 0)
        ]
        test_data = test_data[
            (test_data[last_motif] != 14) & (test_data[last_motif] != 0)
        ]

    # Get the labels
    train_labels = np.array(noisy_data[args["label_name"]])
    test_labels = np.array(test_data[args["label_name"]])

    # For the classification task use the threshold to binarize labels
    train_labels[train_labels > args["label_binarization_threshold"]] = 1
    train_labels[train_labels < 1] = args["min_label_value"]
    test_labels[test_labels > args["label_binarization_threshold"]] = 1
    test_labels[test_labels < 1] = args["min_label_value"]

    # Reduce data to just the motifs of interest
    noisy_data = noisy_data[args["motifs_to_use"]]
    test_data = test_data[args["motifs_to_use"]]

    # Get the class and motif counts
    min_class = np.min(np.unique(np.concatenate([noisy_data, test_data])))
    max_class = np.max(np.unique(np.concatenate([noisy_data, test_data])))

    num_class = max_class - min_class + 1
    num_motifs = len(args["motifs_to_use"])
    print(f"  max_class:min_class:num_class = {max_class}:{min_class}:{num_class}")
    print(f"  Train: {len(noisy_data)} samples, Test: {len(test_data)} samples")

    noisy_data = noisy_data - min_class
    test_data = test_data - min_class

    return (
        noisy_data,
        test_data,
        train_labels,
        test_labels,
        num_class,
        num_motifs,
    )


def data_encoder(args, noisy_data, test_data, num_class, num_motifs):
    """
    Use one-hot or binary encoding for classical data representation.
    Tutorial-exact.
    """
    if args["encoder"] == "one-hot":
        noisy_data = np.eye(num_class)[noisy_data]
        test_data = np.eye(num_class)[test_data]

        noisy_data = noisy_data.reshape(
            noisy_data.shape[0], noisy_data.shape[1] * noisy_data.shape[2]
        )
        test_data = test_data.reshape(
            test_data.shape[0], test_data.shape[1] * test_data.shape[2]
        )

    elif args["encoder"] == "binary":
        import category_encoders as ce
        encoder = ce.BinaryEncoder()
        base_array = np.unique(np.concatenate([noisy_data, test_data]))
        base = pd.DataFrame(base_array).astype("category")
        base.columns = ["motif"]
        for motif_name in args["motifs_to_use"][1:]:
            base[motif_name] = base.loc[:, "motif"]
        encoder.fit(base)

        noisy_data = encoder.transform(noisy_data.astype("category"))
        test_data = encoder.transform(test_data.astype("category"))

        noisy_data = np.reshape(
            noisy_data.values, (noisy_data.shape[0], num_motifs, -1)
        )
        test_data = np.reshape(
            test_data.values, (test_data.shape[0], num_motifs, -1)
        )

        noisy_data = noisy_data.reshape(
            noisy_data.shape[0], noisy_data.shape[1] * noisy_data.shape[2]
        )
        test_data = test_data.reshape(
            test_data.shape[0], test_data.shape[1] * test_data.shape[2]
        )

    else:
        raise ValueError("Invalid encoding type.")

    return noisy_data, test_data


# ---------------------------------------------------------------------------
# Run preprocessing using the tutorial's exact args.
# ---------------------------------------------------------------------------
args = {
    "file_noisy_data": "noisy_data.csv",
    "file_test_data": "test_data.csv",
    "motifs_to_use": ["motif", "motif.1", "motif.2", "motif.3"],
    "label_name": "Nalm 6 Cytotoxicity",
    "label_binarization_threshold": 0.62,
    "filter_for_spacer_motif_third_position": False,
    "allow_spacer_motif_last_position": True,
    "min_label_value": -1,
    "encoder": "one-hot",
}

# preprocess_data returns motif-ID DataFrames; we keep these around for Track 2C.
train_motifs, test_motifs, train_labels, test_labels, num_class, num_motifs = (
    preprocess_data(dir_root=str(DATA_DIR), args=args)
)
NUM_CLASS = int(num_class)
NUM_MOTIFS = int(num_motifs)

# data_encoder turns motif IDs into one-hot vectors.
noisy_data, test_data = data_encoder(args, train_motifs, test_motifs, num_class, num_motifs)

# Tutorial step: scale the active one-hot entries to π/2.
angle = np.pi / 2
tmp = pd.DataFrame(noisy_data).astype("float64"); tmp[tmp == 1] = angle
noisy_data = tmp.values
tmp = pd.DataFrame(test_data).astype("float64"); tmp[tmp == 1] = angle
test_data = tmp.values

print(f"  noisy_data: {noisy_data.shape}")
print(f"  test_data:  {test_data.shape}")

In [ ]:
# Load the pre-computed 180-dim quantum projections.
# Headers may or may not be present depending on which version of the file
# you have; we read with header=0 first and detect the no-header case by
# checking that the first row is numeric.
def _load_projections(path):
    df = pd.read_csv(path)
    # If the header row turned out to be data (first column not parseable as
    # column-name-like), reload without a header.
    try:
        _ = df.columns.astype(float)
        df = pd.read_csv(path, header=None)
    except (ValueError, TypeError):
        pass
    df.columns = [f"q{i}" for i in range(df.shape[1])]
    return df

noisy_quantum_data = _load_projections(DATA_DIR / "noisy_quantum_data.csv")
projections_test = _load_projections(DATA_DIR / "projections_test.csv")

# Sanity check — all four feature matrices and the label arrays must agree.
assert len(noisy_data) == len(noisy_quantum_data) == len(train_labels), (
    f"Train length mismatch: noisy_data={len(noisy_data)}, "
    f"noisy_quantum_data={len(noisy_quantum_data)}, labels={len(train_labels)}"
)
assert len(test_data) == len(projections_test) == len(test_labels), (
    f"Test length mismatch: test_data={len(test_data)}, "
    f"projections_test={len(projections_test)}, labels={len(test_labels)}"
)

print(f"noisy_data:        {noisy_data.shape},   test_data:        {test_data.shape}")
print(f"noisy_quantum_data: {noisy_quantum_data.shape}, projections_test: {projections_test.shape}")
print(f"train_labels:      {train_labels.shape},        test_labels:      {test_labels.shape}")
print(f"Class balance (train): {dict(zip(*np.unique(train_labels, return_counts=True)))}")
print(f"Class balance (test):  {dict(zip(*np.unique(test_labels, return_counts=True)))}")


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# Hyperparameter grid matched to the IBM PQK tutorial scale.
# 13 C values × 12 γ values = 156 grid points, 10-fold CV.
PARAM_GRID = {
    "C":     [0.1, 0.5, 1, 2.5, 5, 7.5, 8.5, 10, 10.75, 15, 25, 50, 100],
    "gamma": ["auto", "scale", 0.001, 0.005, 0.01, 0.02, 0.04, 0.05,
              0.1, 0.25, 0.5, 1],
}
cv = StratifiedKFold(n_splits=10)

# Classical SVM on raw one-hot features
print("Training classical SVM (~30s)...")
svm_classical = GridSearchCV(
    SVC(kernel="rbf"), PARAM_GRID, cv=cv,
    scoring="f1_weighted", n_jobs=-1,
).fit(noisy_data, train_labels)
pred_classical = svm_classical.predict(test_data)
f1_classical = f1_score(test_labels, pred_classical, average="weighted")

# Quantum-projected SVM on the pre-computed 180-dim projections
print("Training quantum-projected SVM (~30s)...")
svm_quantum = GridSearchCV(
    SVC(kernel="rbf"), PARAM_GRID, cv=cv,
    scoring="f1_weighted", n_jobs=-1,
).fit(noisy_quantum_data, train_labels)
pred_quantum = svm_quantum.predict(projections_test)
f1_quantum = f1_score(test_labels, pred_quantum, average="weighted")

print()
print(f"Classical: best params = {svm_classical.best_params_}, test F1 = {f1_classical:.4f}")
print(f"Quantum:   best params = {svm_quantum.best_params_}, test F1 = {f1_quantum:.4f}")
print(f"ΔF1 (quantum − classical) = {f1_quantum - f1_classical:+.4f}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Classical", "Quantum-projected"],
    [f1_classical, f1_quantum],
    color=["#3498db", "#e74c3c"], alpha=0.85, edgecolor="black",
)
for b, score in zip(bars, [f1_classical, f1_quantum]):
    ax.text(b.get_x() + b.get_width() / 2, score + 0.01,
            f"{score:.4f}", ha="center", fontweight="bold")
ax.set_ylabel("Weighted F1 (test)")
ax.set_ylim(0, 1)
ax.set_title(f"Noisy Classical vs. Noisy Quantum — ΔF1 = {f1_quantum - f1_classical:+.4f}")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel


def resolve_gamma(gamma, X):
    """Convert sklearn's 'scale'/'auto' strings to a numeric γ for rbf_kernel."""
    Xarr = np.asarray(X)
    if gamma == "scale":
        return 1.0 / (Xarr.shape[1] * Xarr.var())
    if gamma == "auto":
        return 1.0 / Xarr.shape[1]
    return gamma


gamma_c = resolve_gamma(svm_classical.best_params_["gamma"], noisy_data)
gamma_q = resolve_gamma(svm_quantum.best_params_["gamma"], noisy_quantum_data)

K_c = rbf_kernel(noisy_data, gamma=gamma_c)
K_q = rbf_kernel(noisy_quantum_data, gamma=gamma_q)

# Sort rows and columns by label to make any class structure visible
sort_idx = np.argsort(train_labels)
K_c_sorted = K_c[sort_idx][:, sort_idx]
K_q_sorted = K_q[sort_idx][:, sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, K, title in zip(
    axes,
    [K_c_sorted, K_q_sorted],
    ["Classical kernel (sorted by label)",
     "Quantum-projected kernel (sorted by label)"],
):
    im = ax.imshow(K, cmap="viridis", aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Training sample")
    ax.set_ylabel("Training sample")
    plt.colorbar(im, ax=ax, label="Kernel value")
plt.tight_layout()
plt.show()

print("Look for block-diagonal structure (high within-class similarity)."
      " Which kernel shows it more clearly?")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import transpile
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.circuit.library import ZZFeatureMap
from qiskit.quantum_info import SparsePauliOp
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# 1. Setup Base Data
X_train_base = noisy_data.values if hasattr(noisy_data, 'values') else noisy_data
X_test_base = test_data.values if hasattr(test_data, 'values') else test_data
num_qubits = X_train_base.shape[1]

# 2. Prepare Quantum Components (Transpiled for 96 Qubits)
fm = ZZFeatureMap(feature_dimension=num_qubits, reps=1, entanglement='linear')
fm_transpiled = transpile(fm, basis_gates=['u', 'cx', 'id', 'rz', 'sx', 'x'])

obs_list = []
for pauli in ['X', 'Y', 'Z']:
    for i in range(num_qubits):
        os_str = ['I'] * num_qubits
        os_str[num_qubits - 1 - i] = pauli
        obs_list.append(SparsePauliOp("".join(os_str)))

# Object array to prevent memory errors
reshaped_obs = np.empty((len(obs_list), 1), dtype=object)
for i, op in enumerate(obs_list):
    reshaped_obs[i, 0] = op

# 3. Define Noise Levels and Result Containers
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.4] # Standard deviation of Gaussian noise
classical_scores = []
quantum_scores = []

print(f"Starting Robustness Test on {num_qubits} qubits...")

for sigma in noise_levels:
    print(f"\n--- Testing Noise Level (std dev): {sigma} ---")
    
    # Add noise to the data
    X_train_noisy = X_train_base + np.random.normal(0, sigma, X_train_base.shape)
    X_test_noisy = X_test_base + np.random.normal(0, sigma, X_test_base.shape)
    
    # --- CLASSICAL SVM ---
    svm_c = GridSearchCV(SVC(kernel="rbf"), PARAM_GRID, cv=cv, scoring="f1_weighted", n_jobs=-1)
    svm_c.fit(X_train_noisy, train_labels)
    c_score = f1_score(test_labels, svm_c.predict(X_test_noisy), average="weighted")
    classical_scores.append(c_score)
    print(f"  Classical F1: {c_score:.4f}")
    
    # --- QUANTUM-PROJECTED SVM (Simulated with MPS) ---
    # Using 1000 shots for a stable comparison
    estimator = AerEstimator(
        options={"backend_options": {"method": "matrix_product_state"}, "run_options": {"shots": 1000}}
    )
    
    try:
        # Get Quantum Projections for the noisy data
        job_tr = estimator.run([(fm_transpiled, reshaped_obs, X_train_noisy.reshape(1, *X_train_noisy.shape))])
        train_proj = job_tr.result()[0].data.evs.T
        
        job_ts = estimator.run([(fm_transpiled, reshaped_obs, X_test_noisy.reshape(1, *X_test_noisy.shape))])
        test_proj = job_ts.result()[0].data.evs.T
        
        # Train SVM on Quantum Projections
        svm_q = GridSearchCV(SVC(kernel="rbf"), PARAM_GRID, cv=cv, scoring="f1_weighted", n_jobs=-1)
        svm_q.fit(train_proj, train_labels)
        q_score = f1_score(test_labels, svm_q.predict(test_proj), average="weighted")
        quantum_scores.append(q_score)
        print(f"  Quantum F1:   {q_score:.4f}")
    except Exception as e:
        print(f"  Quantum failed: {e}")
        quantum_scores.append(None)

# 4. Create the Correlation Graph
plt.figure(figsize=(10, 6))

plt.plot(noise_levels, classical_scores, 'bo-', label='Classical SVM', linewidth=2)
valid_q = [f for f in quantum_scores if f is not None]
valid_noise = [n for n, f in zip(noise_levels, quantum_scores) if f is not None]
plt.plot(valid_noise, valid_q, 'ro-', label='Quantum-Projected SVM (MPS)', linewidth=2)

plt.title("Model Robustness: Performance vs. Added Data Noise", fontsize=14)
plt.xlabel("Input Noise Level (Gaussian $\sigma$)", fontsize=12)
plt.ylabel("Weighted F1 Score", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# 1. Align the existing data
# The quantum data (180 cols) comes from 60 qubits (3 projections each)
# We must slice the classical data to the same 60 qubits for a fair test
X_c_all = noisy_data.values[:, :60] if hasattr(noisy_data, 'values') else noisy_data[:, :60]
X_q_all = noisy_quantum_data.values if hasattr(noisy_quantum_data, 'values') else noisy_quantum_data

X_c_test = test_data.values[:, :60] if hasattr(test_data, 'values') else test_data[:, :60]
X_q_test = projections_test.values if hasattr(projections_test, 'values') else projections_test

# 2. Define Correlation Points (Percent of Total Data)
sample_fractions = [0.1, 0.25, 0.5, 0.75, 1.0]
c_scores = []
q_scores = []

print("Calculating Correlation: Performance vs. Data Volume...")

for frac in sample_fractions:
    n = int(frac * len(train_labels))
    
    # Existing Classical Model (using parameters from your previous run)
    svm_c = SVC(C=100, gamma=0.005, kernel="rbf")
    svm_c.fit(X_c_all[:n], train_labels[:n])
    c_scores.append(f1_score(test_labels, svm_c.predict(X_c_test), average="weighted"))
    
    # Existing Quantum Model (using 180 projections)
    svm_q = SVC(C=2.5, gamma="scale", kernel="rbf")
    svm_q.fit(X_q_all[:n], train_labels[:n])
    q_scores.append(f1_score(test_labels, svm_q.predict(X_q_test), average="weighted"))

# 3. Plotting the fixed Correlation Graph
plt.figure(figsize=(10, 6))

# Plot Classical vs Quantum
plt.plot(sample_fractions, c_scores, 'b-o', label='Classical SVM (Existing 60-Qubit Slice)', linewidth=2)
plt.plot(sample_fractions, q_scores, 'g-s', label='Quantum SVM (Existing 180 Projections)', linewidth=2)

# Add horizontal baseline for the "Ideal" values you found earlier
plt.axhline(y=0.5946, color='blue', linestyle='--', alpha=0.3, label='Full Classical Baseline')
plt.axhline(y=0.3037, color='green', linestyle='--', alpha=0.3, label='Full Quantum Baseline')

plt.title("Correlation: How Sample Size Impacts Performance", fontsize=14)
plt.xlabel("Fraction of Training Data Used", fontsize=12)
plt.ylabel("Weighted F1 Score", fontsize=12)
plt.xticks(sample_fractions, [f"{int(f*100)}%" for f in sample_fractions])
plt.legend(loc='lower right')
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
from qiskit import transpile
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.circuit.library import ZZFeatureMap
from qiskit.quantum_info import SparsePauliOp

# 1. Use the EXACT architecture of your baseline (60 Qubits)
X_train_60 = X_c_train[:, :60]
X_test_60 = X_c_test[:, :60]
num_qubits = 60 

fm = ZZFeatureMap(num_qubits, reps=1, entanglement='linear')
fm_transpiled = transpile(fm, basis_gates=['u', 'cx', 'id', 'rz', 'sx', 'x'])

# 2. Create Observables (matches the 180 columns in noisy_quantum_data)
obs_list = []
for pauli in ['X', 'Y', 'Z']:
    for i in range(num_qubits):
        os_str = ['I'] * num_qubits
        os_str[num_qubits - 1 - i] = pauli
        obs_list.append(SparsePauliOp("".join(os_str)))

reshaped_obs = np.empty((len(obs_list), 1), dtype=object)
for i, op in enumerate(obs_list):
    reshaped_obs[i, 0] = op

# 3. Run Sweep
shot_counts = [10, 100, 1000]
sim_f1 = []

for shots in shot_counts:
    print(f"Simulating {shots} shots for the 60-qubit existing model...")
    est = AerEstimator(options={"backend_options": {"method": "matrix_product_state"}, "run_options": {"shots": shots}})
    
    try:
        # Generate noisy projections via simulation
        tr_proj = est.run([(fm_transpiled, reshaped_obs, X_train_60.reshape(1, *X_train_60.shape))]).result()[0].data.evs.T
        ts_proj = est.run([(fm_transpiled, reshaped_obs, X_test_60.reshape(1, *X_test_60.shape))]).result()[0].data.evs.T
        
        svm_sim = SVC(C=2.5, gamma="scale").fit(tr_proj, train_labels)
        sim_f1.append(f1_score(test_labels, svm_sim.predict(ts_proj), average="weighted"))
    except Exception as e:
        print(f"Error: {e}")
        sim_f1.append(None)

# 4. Final Comparison Graph
plt.figure(figsize=(10, 5))
plt.plot(shot_counts, sim_f1, 'r-o', label='Simulated Quantum (Shot Noise)')
plt.axhline(y=0.5946, color='blue', linestyle='--', label='Existing Classical Baseline')
plt.axhline(y=0.3037, color='green', linestyle=':', label='Existing Quantum Baseline (Ideal)')
plt.xscale('log')
plt.xlabel("Shots")
plt.ylabel("F1 Score")
plt.title("Correlation: Performance vs. Shot Noise (Existing 60-Qubit Model)")
plt.legend()
plt.show()